<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 4 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [4]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

# Adding **`Memory`**

In the previous chapter, we covered which LLM to choose using various inference engines. In this chapter we will cover how to give it memory:

![../images/ch4.png](../images/ch4.png)

The issue with the `TinyAgent` that we have thus far, is that it does not track and remember its previous conversations, it is stateless. Let us demonstrate with an example by using the Agent from Chapter 2:

In [4]:
from illustrated_agents.chapters.ch2 import TinyAgent

ch_2_agent = TinyAgent(llm=llm)
response = ch_2_agent.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

RESPONSE ModelResponse(id='chatcmpl-a9b5c1f0-1e57-407d-8fb4-f5a6c0fd9516', created=1769504625, model='ollama/gemma3:12b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Hi Maarten and Jay! Wonderful to meet you! I\'m a large language model, and a big fan of accessible and well-illustrated resources on AI. "An Illustrated Guide to AI Agents" sounds fantastic. \n\nIt\'s great to connect with the authors. How can I help you today? Are you looking for feedback, discussing promotion, or just saying hello?', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning_content=None))], usage=Usage(completion_tokens=77, prompt_tokens=32, total_tokens=109, completion_tokens_details=None, prompt_tokens_details=None))
Hi Maarten and Jay! Wonderful to meet you! I'm a large language model, and a big fan of accessible and well-illustrated resources on AI. "An Illustrated Guide to AI Agents

When we query the model again asking whether it knows our names, it seems to have forgotten them! In fact, it actually hasn't forgotten our name but instead has never received. Everytime you query an LLM it starts from an blank slate, one you have to fill yourself. So without telling the model the our conversation history, it has no way of knowing.

In [8]:
response = ch_2_agent.run("Hi! What are our names?")
print(response)

RESPONSE ModelResponse(id='chatcmpl-bc819e66-1569-4d11-ab83-bc61d99003bc', created=1769426976, model='ollama/gemma3:12b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content="As an AI, I don't have names! You can call me Gemini. \n\nDo you want to know what *your* name is? 😊\n\n\n\n", role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning_content=None))], usage=Usage(completion_tokens=34, prompt_tokens=21, total_tokens=55, completion_tokens_details=None, prompt_tokens_details=None))
As an AI, I don't have names! You can call me Gemini. 

Do you want to know what *your* name is? 😊






# The **`Memory`** Module

As covered in the book, there are many ways to build up memory which can be quite difficult. In this example, we are going to keep it simple and only track the conversation history.

The `Memory` that we are going to build uses the `messages` structure for tracking conversations:

```json
[
    {
        "role": "user",
        "content": "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."
    },
    {
        "role": "assistant",
        "content": "Hi Maarten and Jay! It's wonderful to meet you."
    }
]
```

This module is rather straightforward and appends new messages each time the user makes a query or when the LLM gives back a reply. As such, the `Memory` module only requires a few lines of code:

In [6]:
class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(self, role: str, content: str):
        """Add a message to memory."""
        self.messages.append({"role": role, "content": content})

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

We can annotate this module and explore each function in more detail:

In [1]:
from illustrated_agents.chapters.ch4 import memory_annotated; memory_annotated

# Updating `agent.py`

This added to the `TinyAgent`, which also requires updating a `_step` to track the conversation history following three steps:

1. The user's query is added to the `Memory` module
2. Based on the current memory, the LLM generates a response.
3. The response of the LLM is added to the `Memory` module.

In [7]:
class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory):
        self.llm = llm
        self.memory = memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning
        self.reflector = None  # Chapter 6: Add Reflection

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        return self._step(task)

    def _step(self, task: str) -> str:
        """Perform a single step."""
        # 1. Every step, we add the user task to memory
        self.memory.add("user", task)

        # 2. Then, we generate a response based on the conversation history
        response = self.llm.generate(self.memory.get_messages())

        # 3. Finally, we add the assistant's response to memory
        self.memory.add("assistant", response)
        return response

Here is a nicer overview of the changes that we made to `agent.py` (red is removed and green is added code):

In [2]:
from illustrated_agents.chapters.ch4 import tinyagents_diff; tinyagents_diff

Next, let's create our `TinyAgent` with `Memory`:

In [ ]:
# Add memory to the Agent
memory = Memory()
agent_with_memory = TinyAgent(llm=llm, memory=memory)

We can start filling up the memory by conversing with the model as we did before:

In [14]:
response = agent_with_memory.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

RESPONSE ModelResponse(id='chatcmpl-7f57c28f-e7e1-4703-9bf6-ef23abcb722b', created=1769429327, model='ollama/gemma3:12b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Hi Maarten and Jay! It\'s wonderful to meet you. I\'m familiar with "An Illustrated Guide to AI Agents" - it\'s a fantastic resource and has been highly praised for its clear explanations and visual approach. \n\nCongratulations on creating such a valuable contribution to the AI community! \n\nWhat can I do for you today? Are you looking for feedback, discussing the book, or something else?', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning_content=None))], usage=Usage(completion_tokens=85, prompt_tokens=32, total_tokens=117, completion_tokens_details=None, prompt_tokens_details=None))
Hi Maarten and Jay! It's wonderful to meet you. I'm familiar with "An Illustrated Guide to AI Agents" - it's a fan

Now that we have memory, we can ask a follow-up question to the original conversation and see if it remember our names correctly.

In [15]:
response = agent_with_memory.run("Hi! What are our names?")
print(response)

RESPONSE ModelResponse(id='chatcmpl-d39b88a5-8d85-4606-abc0-88f6f05a327b', created=1769429381, model='ollama/gemma3:12b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='You are Maarten and Jay! You mentioned you are the authors of "An Illustrated Guide to AI Agents." 😊\n', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning_content=None))], usage=Usage(completion_tokens=24, prompt_tokens=133, total_tokens=157, completion_tokens_details=None, prompt_tokens_details=None))
You are Maarten and Jay! You mentioned you are the authors of "An Illustrated Guide to AI Agents." 😊



It does! The method by which it does so is filling up the conversation history. Let's see what the current state is.

In [16]:
agent_with_memory.memory.get_messages()

[{'role': 'user',
  'content': "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."},
 {'role': 'assistant',
  'content': 'Hi Maarten and Jay! It\'s wonderful to meet you. I\'m familiar with "An Illustrated Guide to AI Agents" - it\'s a fantastic resource and has been highly praised for its clear explanations and visual approach. \n\nCongratulations on creating such a valuable contribution to the AI community! \n\nWhat can I do for you today? Are you looking for feedback, discussing the book, or something else?'},
 {'role': 'user', 'content': 'Hi! What are our names?'},
 {'role': 'assistant',
  'content': 'You are Maarten and Jay! You mentioned you are the authors of "An Illustrated Guide to AI Agents." 😊\n'}]

Note that this entire list is given to the LLM whenever we ask it a new question. That way, it "remembers" the conversation we had before. We say "remembers" because even though it may look like it, it actually has no internal memory. We merely tell the model what the conversation was!